In [1]:
import numpy as np
import librosa
import joblib
import os
from scipy.signal import butter, lfilter
import pandas as pd

In [2]:


# === Parameters for STFT feature extraction ===
SAMPLE_RATE = 22050
N_FFT = 1024
HOP_LENGTH = 512
DURATION = 3  # seconds
N_FRAMES = int((SAMPLE_RATE * DURATION) / HOP_LENGTH) + 1
N_BINS = N_FFT // 2 + 1

# === Bandpass filter ===
def bandpass_filter(data, sr, lowcut=100, highcut=4000):
    nyq = 0.5 * sr
    b, a = butter(4, [lowcut / nyq, highcut / nyq], btype='band')
    return lfilter(b, a, data)

# === Feature extraction: STFT summary ===
def extract_stft_features(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE, duration=DURATION)
    y = bandpass_filter(y, sr)

    stft = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)
    spectrogram = np.abs(stft)
    log_stft = librosa.amplitude_to_db(spectrogram, ref=np.max)

    if log_stft.shape[1] < N_FRAMES:
        pad_width = N_FRAMES - log_stft.shape[1]
        log_stft = np.pad(log_stft, ((0, 0), (0, pad_width)), mode='constant')
    elif log_stft.shape[1] > N_FRAMES:
        log_stft = log_stft[:, :N_FRAMES]

    # Summary features
    mean = np.mean(log_stft, axis=1)
    std = np.std(log_stft, axis=1)
    min_ = np.min(log_stft, axis=1)
    max_ = np.max(log_stft, axis=1)
    
    X_df = pd.DataFrame(np.hstack([mean, std, min_, max_]).reshape(1, -1))  # shape: (1, 2052)

    return X_df
    

# === Two-stage classifier ===
def classify_audio(audio_path):
    # Step 1: Extract features
    all_features = extract_stft_features(audio_path)
    features_boat = joblib.load("features_boat.joblib")
    features_type = joblib.load("features_type.joblib")
    
    # Step 2: Load and apply first model (Boat vs. No Boat)
    model_boat = joblib.load('lightgbm_model_boat_no_boat.joblib')
    X_boat = all_features[features_boat]
    is_boat = model_boat.predict(X_boat)[0]

    if is_boat == 0:
        return "Not a boat"
    else:
        # Step 3: Apply second model (Speedboat vs. Other boat)
        model_type = joblib.load('lightgbm_model_if_speedboat.joblib')
        X_type = all_features[features_type]
        is_speedboat = model_type.predict(X_type)[0]
        return "Speedboat" if is_speedboat == 1 else "Other boat"




In [12]:
# === Example usage ===
test_file_1 = "/Users/silent360/Meng_Study/GMU/DAEN690/Data/QiandaoEar22/label_1/boat_1/20220624102845_M_SpeedBoat_M_M&Helicopter_F_W_0_label__1__8_1.wav"
test_file_2 = "/Users/silent360/Meng_Study/GMU/DAEN690/Data/QiandaoEar22/label_1/boat_0/20220624162953_M_UUV_M_M&MotorBoat_M_M&ArtificialSignals_0_label__1__13_1.wav"
test_file_3 = "/Users/silent360/Meng_Study/GMU/DAEN690/Data/QiandaoEar22/label_0/20220624102345_S_SpeedBoat_N_S_1_label__0__23.wav"



In [13]:
result1 = classify_audio(test_file_1)
print("Prediction:", result1)

[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
Prediction: Speedboat


In [8]:
result2 = classify_audio(test_file_2)
print("Prediction:", result2)

[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
Prediction: Other boat


In [9]:
result3 = classify_audio(test_file_3)
print("Prediction:", result3)

[LightGBM] [Warning] lambda_l1 is set=1.0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0
[LightGBM] [Warning] lambda_l2 is set=0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0
Prediction: Not a boat
